In [9]:
#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()


#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()




In [10]:
base2 = pd.read_sql_query(f"SELECT * FROM MMEME10", con=connection2)
a=base2[base2.duplicated(subset=["emecod"])]
print(a)

base2.to_sql(name='tmp_mmeme10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mmeme10 FALSO CON LO OBTENIDO DEL ORACLE
tabla='mmeme10'
query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mmeme10 
ALTER COLUMN emecod TYPE CHARACTER (2),
ALTER COLUMN emedes TYPE CHARACTER (30);


UPDATE mmeme10 
SET 

emedes= t.emedes


FROM tmp_mmeme10 t 
WHERE mmeme10.emecod = t.emecod and mmeme10.emecod IS NOT NULL and t.emecod IS NOT NULL ;

INSERT INTO mmeme10 
(emecod, emedes) 

SELECT 
emecod, emedes

FROM tmp_mmeme10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mmeme10 
    WHERE mmeme10.emecod = tmp_mmeme10.emecod and mmeme10.emecod IS NOT NULL and tmp_mmeme10.emecod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mmeme10;
DELETE FROM mmeme10 WHERE emecod ='';
"""


c2= text(query2)
cursor=connection3.execute(c2)


query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


base2 = pd.read_sql_query(f"SELECT * FROM mmeme10", con=connection3)



Empty DataFrame
Columns: [emecod, emedes]
Index: []
Cantidad de filas en la tabla mmeme10 antes de la inserción: 13
Cantidad de filas en la tabla mmeme10 después de la inserción: 13
La cantidad de filas insertadas fue de: 0


In [11]:
#################################################################################################################################################################################################################################################################################

base2.to_sql(name='tmp_mmeme10', con=engine4, if_exists='replace', index=False)


tabla='dim_emecod'
query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")



query="""

ALTER TABLE tmp_mmeme10 
ALTER COLUMN emecod TYPE CHARACTER (2),
ALTER COLUMN emedes TYPE CHARACTER (30);


INSERT INTO dim_emecod (cod_eme, des_eme) 
SELECT emecod, emedes  

FROM tmp_mmeme10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emecod 
    WHERE dim_emecod.cod_eme = tmp_mmeme10.emecod
);

DROP TABLE tmp_mmeme10;
"""

c1 = text(query)
cursor = connection4.execute(c1)

query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection4.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")



Cantidad de filas en la tabla dim_emecod antes de la inserción: 13
Cantidad de filas en la tabla dim_emecod después de la inserción: 13
La cantidad de filas insertadas fue de: 0


In [12]:
connection4.close()
connection3.close()
connection2.close()
